# Setting up the environment

### Importing libraries:

* langchain-openai: langchain-openai: A LangChain integration that enables seamless interaction with OpenAI's models for advanced language processing and AI workflows.</br>
* langchain.prompts import PromptTemplate:A utility in LangChain for creating dynamic, reusable text prompts with placeholders for structured AI interactions.</br>
* requests: A popular library for making HTTP requests in Python.</br>
* json: Used for parsing JSON data, which is common in API responses.</br>
* from google.colab : Used to securely store and retrieve user-specific data in Google Colab.


In [3]:
%pip install langchain
%pip install langchain-openai
from langchain_openai import ChatOpenAI
from langchain.prompts import PromptTemplate
from google.colab import userdata

# Secure API key handling

In [10]:
OPENAI_KEY = userdata.get("OPENAI_API_KEY")  # Ensure this is set in the environment
LLUMOAI_KEY = userdata.get("LLUMOAI_API_KEY")  # Ensure this is set in the environment(Visit https://app.llumo.ai/ to get your own Api key)
if not OPENAI_KEY:
    raise ValueError("Missing OpenAI API key. Set OPENAI_API_KEY as an environment variable.")


chatModel = ChatOpenAI(model="gpt-4", openai_api_key=OPENAI_KEY)


# Defining Prompt

In [18]:
# Defining the Prompt
promptTemplate = PromptTemplate.from_template("Explain {topic} in a simple and easy-to-understand way.")


# Creating Chain

In [ ]:
# Create a chain
chain = promptTemplate | chatModel

# Getting Respone

In [15]:
# Run the chain with a topic
topic = "quantum physics"
response = chain.invoke({"topic": topic})

In [22]:
response.content

"Quantum physics, also known as quantum mechanics, is a part of physics that deals with events on a very small scale, like atoms and particles. Here's a simple breakdown:\n\n1. Particle-Wave Duality: Everything in the universe has both particle and wave properties. Think of how light can behave as both a particle and a wave.\n\n2. Quantum Superposition: Particles can be in multiple places at the same time. Imagine if you could be at home and at school at the same moment until someone observed your location!\n\n3. Quantum Entanglement: Two particles can be linked despite being far apart and remain instantaneously connected to each other's state. If one changes, the other one will too, no matter how far apart they are.\n\n4. Uncertainty Principle: The more accurately you know the position of a particle, the less exactly you can know its speed and vice versa. In other words, we can't measure both perfectly at the same time.\n\n5. Quantum Tunneling: Particles can pass through barriers they

# Evaluating Responses with Llumo

### Evaluate the OpenAI Response:
* We evaluate the response generated by the OpenAI API using the Llumo evaluation Api.</br>
* We call the Api the example prompt and the openai_output.</br>
* We check if the evaluation was successful:</br>
* If successful, we print the Llumo evaluation results.</br>
* If the evaluation fails, we print an error message and indicate that the original prompt can be used if evaluation fails.

In [23]:
import requests
# Define the endpoint, headers, and payload
LLUMO_ENDPOINT = "https://app.llumo.ai/api/create-eval-analytics"
headers = {
    "Authorization": f"Bearer {LLUMOAI_KEY}", # Replace with your LLumo API key it will look like this "Bearer A1B2C3"
    "Content-Type": "application/json"
}
payload = {
    "prompt": promptTemplate.template,
    "input": {"topic":"quantum physics"},
    "output": response.content,
    "analytics": ["Confidence"] # ANALYTICS NAME are Confidence,Clarity,Context.....etc.
}
# Make the API request
response = requests.post(LLUMO_ENDPOINT, json=payload, headers=headers)
print(response)
try:
    result = response.json()  # Parse the JSON response
    print("statusCode : ", result['data']['statusCode'])
    print("message : ",result['data']['message'])
    # Extract the 'data' part
    data = result.get('data', {})
    print("Analytics:", data)
    # Return the data and a success flag

except Exception as e:
  print(e)

<Response [200]>
statusCode :  200
message :  SUCCESS
Analytics: {'data': '{"analyticsScore": {"confidence": 65, "context": 75, "clarity": 80, "overallScore": 73}, "reasoning": {"confidence": ["The output demonstrates confidence by presenting complex concepts in a clear and structured manner.", "The use of analogies and examples enhances the clarity and confidence of the explanation.", "While the output doesn\'t explicitly state certainty with phrases like \'I am certain...\', the overall tone and style project confidence in the presented information.", "However, the output\'s length exceeds the 20-word limit in the prompt, slightly detracting from the overall confidence score."], "context": ["The output directly addresses the prompt\'s request to explain quantum physics simply.", "It covers key concepts of quantum physics, demonstrating contextual understanding.", "The explanations are logically presented and coherent.", "The use of analogies helps to make the complex concepts more ac

In [24]:
data

{'data': '{"analyticsScore": {"confidence": 65, "context": 75, "clarity": 80, "overallScore": 73}, "reasoning": {"confidence": ["The output demonstrates confidence by presenting complex concepts in a clear and structured manner.", "The use of analogies and examples enhances the clarity and confidence of the explanation.", "While the output doesn\'t explicitly state certainty with phrases like \'I am certain...\', the overall tone and style project confidence in the presented information.", "However, the output\'s length exceeds the 20-word limit in the prompt, slightly detracting from the overall confidence score."], "context": ["The output directly addresses the prompt\'s request to explain quantum physics simply.", "It covers key concepts of quantum physics, demonstrating contextual understanding.", "The explanations are logically presented and coherent.", "The use of analogies helps to make the complex concepts more accessible, enhancing contextual relevance.", "While the output is 